# 🎓 Machine Learning — Ultimate Cheat Sheet
**Open Book Exam Reference | ATU Galway | Donny Hurley**

---
## Table of Contents
1. [Imports & Setup](#1)
2. [ML Fundamentals](#2)
3. [Generalisation, Overfitting & Underfitting](#3)
4. [Train / Validation / Test Splits & Cross-Validation](#4)
5. [Bias–Variance Tradeoff & Regularisation](#5)
6. [Metrics — Regression](#6)
7. [Metrics — Classification](#7)
8. [Feature Scaling & Pipelines](#8)
9. [Linear Regression](#9)
10. [Logistic Regression](#10)
11. [k-Nearest Neighbours (kNN)](#11)
12. [Decision Trees](#12)
13. [Random Forests](#13)
14. [Support Vector Machines (SVM)](#14)
15. [Multi-class Strategies](#15)
16. [GridSearchCV — Hyperparameter Tuning](#16)
17. [K-Means Clustering (Unsupervised)](#17)
18. [Complete Workflow Template](#18)
19. [Quick Reference: Pros & Cons Table](#19)

---
<a id='1'></a>
## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Preprocessing ---
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

# --- Model Selection ---
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# --- Models ---
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans

# --- Metrics ---
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)

# --- Datasets ---
from sklearn.datasets import load_iris, load_breast_cancer, make_classification, make_regression

print("All imports successful!")

---
<a id='2'></a>
## 2. ML Fundamentals

### What is Machine Learning?
Algorithms that **automatically improve performance through experience**.

> **Core idea:** `y = f(x)` — we use data to find the best `f` (the model).

### Types of Learning

| Type | Description | Examples |
|------|-------------|----------|
| **Supervised** | We know the correct output `y` for each input `x` | Regression, Classification |
| **Unsupervised** | No labels — find hidden structure | k-Means, PCA |
| **Reinforcement** | Agent learns from rewards (not covered in this course) | Games, Robotics |

### Supervised Learning Types
- **Regression** — predict a **continuous** value (e.g. house price, cooking time)
- **Classification** — predict a **discrete class/category** (e.g. spam/not spam, digit 0-9)

### sklearn API Pattern — Same for Every Model
```python
model = SomeModel(hyperparameters)   # 1. Initialise
model.fit(X_train, y_train)           # 2. Train
y_pred = model.predict(X_test)        # 3. Predict
score  = model.score(X_test, y_test)  # 4. Evaluate
```

### Data Structure
- `X` → **2D matrix** — shape `(n_samples, n_features)`. Each **row** = one sample, each **column** = one feature.
- `y` → **1D vector** — shape `(n_samples,)`. The correct answer for each sample.
- Both should be **numpy arrays**.

In [ ]:
# Data structure demo
X = np.array([[1.5, 2.0],   # sample 1: feature1=1.5, feature2=2.0
              [3.0, 4.5],   # sample 2
              [2.1, 1.8]])  # sample 3
y = np.array([0, 1, 0])    # labels for each sample

print(f"X shape: {X.shape}  → ({X.shape[0]} samples, {X.shape[1]} features)")
print(f"y shape: {y.shape}  → ({y.shape[0]} labels)")

---
<a id='3'></a>
## 3. Generalisation, Overfitting & Underfitting

**Generalisation** = how well the model performs on **previously unseen data**. This is the goal of ML.

### Overfitting
- Model learns the **noise** in the training data, not the underlying pattern.
- Training loss → 0, but test loss is high.
- Like a student who memorises the answers — fails on new questions.
- **Low bias, high variance**
- Signs: training score >> validation score

### Underfitting
- Model is **too simple** to capture the data's trend.
- Can't even reduce training loss to something reasonable.
- **High bias, low variance**
- Signs: both training and validation score are poor

### Fixes

| Problem | Fix |
|---------|-----|
| Underfitting | More complex model, train longer, better loss function |
| Overfitting | Simpler model, fewer features, **more data**, **regularisation** |

### Polynomial Degree Example
```
M=1 (linear)    → underfit (high bias)
M=3 or 4        → just right
M=9             → overfit (training error=0, test error huge)
```

In [ ]:
# Visualise overfitting/underfitting with polynomial degrees
np.random.seed(42)
X_demo = np.sort(np.random.rand(20, 1) * 6 - 3, axis=0)
y_demo = np.sin(X_demo).ravel() + np.random.randn(20) * 0.3

X_plot = np.linspace(-3, 3, 300).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
degrees = [1, 4, 15]
titles  = ['Underfit (degree=1)', 'Good fit (degree=4)', 'Overfit (degree=15)']

for ax, deg, title in zip(axes, degrees, titles):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_demo, y_demo)
    ax.scatter(X_demo, y_demo, color='black', s=20, label='Data')
    ax.plot(X_plot, model.predict(X_plot), color='red', linewidth=2)
    train_r2 = model.score(X_demo, y_demo)
    ax.set_title(f'{title}\nTrain R²={train_r2:.3f}')
    ax.set_ylim(-2, 2)

plt.tight_layout()
plt.suptitle('Overfitting vs Underfitting', fontsize=14, y=1.02)
plt.show()

---
<a id='4'></a>
## 4. Train / Validation / Test Splits & Cross-Validation

### The Three Sets
| Set | Purpose | Rule |
|-----|---------|------|
| **Training** | Fit model parameters | Used in `.fit()` |
| **Validation** | Tune hyperparameters, select model | Used for decisions |
| **Test** | Final honest evaluation — **use ONCE at the very end** | Never used for decisions |

> ⚠️ **The test set is sacred.** Using it to make any decision contaminates it.

### Typical Splits
- 2-way: 80% train / 20% test  
- 3-way: 60% train / 20% val / 20% test (or 70/15/15)

### k-Fold Cross-Validation
Split training data into `k` folds. Train on `k-1` folds, validate on the remaining fold. Repeat `k` times. **Average the scores.**
- Reduces bias from a single validation split.
- Default `k=5` in sklearn.
- Not feasible for deep learning (too slow).

### Procedure
1. Split data → train + test (keep test locked away)
2. Use cross-validation on training data to tune hyperparameters
3. Select best model / hyperparameters
4. (Optional) Retrain on all training data with best hyperparameters
5. Evaluate **once** on test set

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score

# --- 2-way split ---
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

# --- 3-way split ---
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# --- k-fold cross-validation ---
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
cv_scores = cross_val_score(model, X_train_full, y_train_full, cv=5)  # 5-fold
print(f"\nCV scores: {cv_scores}")
print(f"Mean CV score: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

---
<a id='5'></a>
## 5. Bias–Variance Tradeoff & Regularisation

### Bias–Variance Tradeoff
| | Bias | Variance | Result |
|--|------|----------|--------|
| **Underfit** | High | Low | Bad on both train & test |
| **Just right** | Medium | Medium | Good generalisation |
| **Overfit** | Low | High | Great on train, bad on test |

**Bias** = model too simple, ignores data.  
**Variance** = model too sensitive, reacts to noise.

### Regularisation
Adds a **penalty term** to the loss function to discourage large weights → prevents overfitting.

$$L(w) = \text{MSE}(w) + \lambda \sum_{j=1}^{n} w_j^2$$

| Method | Penalty | sklearn | Effect |
|--------|---------|---------|--------|
| **L2 / Ridge** | $\lambda \sum w_j^2$ | `Ridge(alpha=λ)` | Shrinks all weights; never zeroes |
| **L1 / Lasso** | $\lambda \sum |w_j|$ | `Lasso(alpha=λ)` | Can zero-out features (feature selection) |
| **Elastic Net** | L1 + L2 | `ElasticNet` | Combination of both |

**λ (alpha/C):**
- Large λ → heavy penalty → weights shrink → underfitting risk
- Small λ → light penalty → overfitting risk

> In `SVC`, the regularisation parameter is **C** (inverse of λ):  
> Small C = large margin, allows violations | Large C = hard margin, small violations

In [ ]:
from sklearn.linear_model import Ridge, Lasso

X_reg, y_reg = make_regression(n_samples=100, n_features=20, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

alphas = [0.001, 0.01, 0.1, 1, 10, 100]
print(f"{'Alpha':<10} {'Ridge Train R²':<18} {'Ridge Val R²':<18} {'Lasso Val R²'}")
print("-" * 60)
for a in alphas:
    ridge = Ridge(alpha=a).fit(X_tr, y_tr)
    lasso = Lasso(alpha=a, max_iter=5000).fit(X_tr, y_tr)
    print(f"{a:<10} {ridge.score(X_tr,y_tr):<18.4f} {ridge.score(X_te,y_te):<18.4f} {lasso.score(X_te,y_te):.4f}")

---
<a id='6'></a>
## 6. Metrics — Regression

| Metric | Formula | Range | Good when |
|--------|---------|-------|----------|
| **MSE** | $\frac{1}{m}\sum(\hat{y}_i - y_i)^2$ | 0 → ∞ (lower better) | Standard loss |
| **RMSE** | $\sqrt{\text{MSE}}$ | 0 → ∞ (lower better) | Same units as y |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | (-∞, 1] (1 = perfect, 0 = baseline, <0 = worse than baseline) | Proportion of variance explained |

- For `LinearRegression`, `.score()` returns **R²**
- MSE: closer to 0 the better
- R²: closer to 1 the better

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

X_r, y_r = make_regression(n_samples=200, n_features=1, noise=15, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_tr, y_tr)
y_pred = lr.predict(X_te)

mse  = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, y_pred)

print(f"MSE  : {mse:.4f}   (closer to 0 = better)")
print(f"RMSE : {rmse:.4f}   (closer to 0 = better, same units as y)")
print(f"R²   : {r2:.4f}   (closer to 1 = better)")
print(f"\n.score() = {lr.score(X_te, y_te):.4f}  ← this is R² for LinearRegression")
print(f"\nIntercept (w0): {lr.intercept_:.4f}")
print(f"Coefficients  : {lr.coef_}")

---
<a id='7'></a>
## 7. Metrics — Classification

### Confusion Matrix (Binary)
```
                  Predicted
               Pos      Neg
Actual Pos  [ TP    |   FN ]
       Neg  [ FP    |   TN ]
```
- **TP** = True Positive (correctly predicted positive)
- **TN** = True Negative (correctly predicted negative)
- **FP** = False Positive (predicted positive, actually negative) — Type I error
- **FN** = False Negative (predicted negative, actually positive) — Type II error

### Key Metrics
| Metric | Formula | Read from confusion matrix |
|--------|---------|----------------------------|
| **Accuracy** | $\frac{TP+TN}{TP+FP+TN+FN}$ | Overall correctness |
| **Precision** | $\frac{TP}{TP+FP}$ | Read from **column** (of predicted) |
| **Recall** | $\frac{TP}{TP+FN}$ | Read from **row** (of actual) |
| **F1-Score** | $2 \cdot \frac{P \cdot R}{P+R}$ | Harmonic mean of P and R |

### When to Use What
- **Accuracy** — works well when classes are **balanced**
- **F1-Score** — better for **imbalanced classes** (e.g. cancer detection)
- **Recall** — prioritise when **false negatives are costly** (missing cancer)
- **Precision** — prioritise when **false positives are costly** (spam filter)

> **Extreme example:** A model that always predicts "no cancer" gets 99.5% accuracy but misses every cancer case. F1 = 0.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)

X_c, y_c = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_tr, y_tr)
y_pred = model.predict(X_te)

print(f"Accuracy  : {accuracy_score(y_te, y_pred):.4f}")
print(f"Precision : {precision_score(y_te, y_pred):.4f}")
print(f"Recall    : {recall_score(y_te, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_te, y_pred):.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_te, y_pred, target_names=['Malignant', 'Benign']))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix(y_te, y_pred),
                       display_labels=['Malignant', 'Benign']).plot(ax=ax)
ax.set_title('Confusion Matrix')
plt.show()

In [ ]:
# Manual confusion matrix calculation
y_true = [2, 0, 2, 2, 0, 1]
y_pred_ex = [0, 0, 2, 2, 0, 2]
cm = confusion_matrix(y_true, y_pred_ex)
print("Confusion matrix:")
print(cm)
print("\nDiagonals = correct predictions")
print("Off-diagonals = misclassifications")
# Precision from columns, Recall from rows

---
<a id='8'></a>
## 8. Feature Scaling & Pipelines

### Why Scale?
Distance-based models (kNN, SVM, k-Means) are **very sensitive to feature scale**. A feature ranging 0–1000 will dominate one ranging 0–1.

| Scaler | What it does | Formula | Use when |
|--------|-------------|---------|----------|
| **MinMaxScaler** | Scales to [0, 1] | $x' = \frac{x - x_{min}}{x_{max} - x_{min}}$ | Known range, no outliers |
| **StandardScaler** | Mean=0, Std=1 | $x' = \frac{x - \mu}{\sigma}$ | General purpose, handles outliers better |

### Pipeline — The Right Way
**Never fit the scaler on test data.** Use a pipeline: scaling is part of the model.

```python
model = make_pipeline(
    StandardScaler(),              # Step 1: scale
    KNeighborsClassifier(k=9)     # Step 2: classify
)
model.fit(X_train, y_train)  # scaler fits on train data only
model.score(X_test, y_test)  # scaler transforms test data using train stats
```

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline

data = np.array([[1000, 0.5], [1500, 0.8], [800, 0.2], [2000, 1.1]])

mm = MinMaxScaler()
ss = StandardScaler()

print("Original:\n", data)
print("\nMinMaxScaler (range 0-1):\n", mm.fit_transform(data))
print("\nStandardScaler (mean=0, std=1):\n", ss.fit_transform(data).round(3))

# Pipeline example — correct way
X_c, y_c = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9))
pipe.fit(X_tr, y_tr)
print(f"\nkNN with StandardScaler: {pipe.score(X_te, y_te):.4f}")

---
<a id='9'></a>
## 9. Linear Regression

### Theory
Predict a **continuous** output from features using a linear combination:
$$\hat{y} = w_0 + w_1x_1 + w_2x_2 + \ldots + w_nx_n$$

- **Simple:** one feature → $\hat{y} = w_0 + w_1x$
- **Multiple:** many features
- **Polynomial:** $\hat{y} = w_0 + w_1x + w_2x^2 + \ldots$ (still linear in the weights!)

### Training (Loss Function)
Minimise **Mean Squared Error**:
$$L(w) = \frac{1}{2m}\sum_{i=1}^{m}(\hat{y}_i - y_i)^2$$

Parameters are found analytically (OLS / Ordinary Least Squares) or via gradient descent.

### Key Parameters
- `lr.coef_` → weights $w_1, w_2, \ldots$ (one per feature)
- `lr.intercept_` → bias term $w_0$
- `.score()` → R² score

### Pros & Cons
| ✅ Pros | ❌ Cons |
|---------|--------|
| Simple, interpretable | Assumes linear relationship |
| Fast to train | Sensitive to outliers |
| Works well for linear data | Extrapolation is risky |
| No hyperparameters (basic form) | Can't capture non-linear patterns |

> ⚠️ Always plot a scatterplot first. If the data is not scattered about a line, do **not** use linear regression.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# --- Simple Linear Regression ---
X_r, y_r = make_regression(n_samples=100, n_features=1, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_tr, y_tr)

y_pred = lr.predict(X_te)
print(f"Intercept (w0): {lr.intercept_:.4f}")
print(f"Coefficients  : {lr.coef_}")
print(f"MSE           : {mean_squared_error(y_te, y_pred):.4f}")
print(f"R²            : {r2_score(y_te, y_pred):.4f}")
print(f".score()      : {lr.score(X_te, y_te):.4f}  ← same as R²")

plt.scatter(X_te, y_te, alpha=0.7, label='Actual')
plt.plot(X_te, y_pred, color='red', label='Predicted')
plt.title('Linear Regression')
plt.legend()
plt.show()

# --- Multiple Features ---
X_multi, y_multi = make_regression(n_samples=200, n_features=3, noise=10, random_state=42)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)
lr2 = LinearRegression().fit(X_tr2, y_tr2)
print(f"\nMultiple regression R²: {lr2.score(X_te2, y_te2):.4f}")
print(f"Weights: {lr2.coef_}")

# --- Polynomial Regression ---
poly_model = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
poly_model.fit(X_tr, y_tr)
print(f"\nPolynomial (degree=3) R²: {poly_model.score(X_te, y_te):.4f}")

---
<a id='10'></a>
## 10. Logistic Regression

### Theory
Despite the name — **Logistic Regression is a classification algorithm.**

Uses the **sigmoid function** to squash output to [0, 1]:
$$\hat{y} = \frac{1}{1 + e^{-(w_0 + w_1x_1 + \ldots + w_nx_n)}}$$

Output is a **probability**. Apply a **decision boundary** (default 0.5):
- $\hat{p} \geq 0.5$ → Class 1
- $\hat{p} < 0.5$ → Class 0

### Why Not Use Linear Regression for Classification?
- Can predict values < 0 or > 1 (meaningless as probabilities)
- Outliers distort the fit badly
- MSE gives a non-convex loss for classification

### Loss Function (Log-Loss / Binary Cross-Entropy)
$$L(w) = -\frac{1}{m}\sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}) + (1-y^{(i)}) \log(1-\hat{y}) \right]$$

### Key Parameters
- `penalty='l2'` (default) — regularisation type
- `C` — inverse regularisation strength (default=1). Smaller C = more regularisation
- `max_iter=100` — increase if model doesn't converge
- `model.predict_proba(X)` — get probabilities for all classes
- `model.classes_` — list of class labels the model knows

### Pros & Cons
| ✅ Pros | ❌ Cons |
|---------|--------|
| Simple, fast, interpretable | Only linear decision boundary |
| Returns probabilities | Assumes linear separability |
| Built-in L2 regularisation | Bad with many irrelevant features |
| Works for multi-class via softmax | Poor with non-linear problems |

In [ ]:
from sklearn.linear_model import LogisticRegression

X_c, y_c = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Build with pipeline (scaling important for logistic regression)
log_model = make_pipeline(
    MinMaxScaler(),
    LogisticRegression(max_iter=1000)  # increase max_iter if ConvergenceWarning
)
log_model.fit(X_tr, y_tr)

print(f"Accuracy: {log_model.score(X_te, y_te):.4f}")

# Access inner model
inner = log_model.named_steps['logisticregression']
print(f"Classes: {inner.classes_}")
print(f"Coef shape: {inner.coef_.shape}")  # (n_classes, n_features)

# Probabilities
probs = log_model.predict_proba(X_te[:5])
print(f"\nProbabilities for first 5 samples:")
print(probs.round(3))
print(f"Predictions: {log_model.predict(X_te[:5])}")

print("\n--- Classification Report ---")
print(classification_report(y_te, log_model.predict(X_te),
                             target_names=['Malignant', 'Benign']))

---
<a id='11'></a>
## 11. k-Nearest Neighbours (kNN)

### Theory
**"Tell me who your neighbours are, and I'll tell you who you are."**

**Algorithm:**
1. For a new point, calculate the distance to **every** training point
2. Find the `k` closest points (neighbours)
3. **Classification:** majority vote among k neighbours
4. **Regression:** average the k neighbours' values

**Distance:** Euclidean distance (default):
$$d(t, s) = \sqrt{(t_1-s_1)^2 + (t_2-s_2)^2 + \ldots + (t_p-s_p)^2}$$

### Lazy vs Eager Learning
- **kNN is lazy** — stores all training data, does nothing during "training", works at prediction time
- **Eager** models (LR, SVM, RF) — generalise before seeing the query
- Lazy = more memory, slow prediction; Eager = less memory, fast prediction

### Choosing k
- Small k → overfitting (noisy boundary)
- Large k → underfitting (ignores local patterns)
- Use **cross-validation** to find the best k
- Rule of thumb: use an **odd** k to avoid tie votes

### Curse of Dimensionality
In high dimensions, all points become equidistant → kNN fails.
**Fix:** Use PCA or StandardScaler before kNN.

### Pros & Cons
| ✅ Pros | ❌ Cons |
|---------|--------|
| Simple, intuitive | Slow prediction (checks all training points) |
| No training time | High memory (stores all data) |
| Non-linear boundary | Sensitive to irrelevant features |
| Works for regression & classification | Scale matters — MUST scale features |
| | Curse of dimensionality |

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_c, y_c = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Without scaling — often worse
knn_no_scale = KNeighborsClassifier(n_neighbors=9)
knn_no_scale.fit(X_tr, y_tr)
print(f"kNN (no scaling)  : {knn_no_scale.score(X_te, y_te):.4f}")

# With scaling — much better
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9))
knn_scaled.fit(X_tr, y_tr)
print(f"kNN (StandardScaler): {knn_scaled.score(X_te, y_te):.4f}")

# Finding the best k using cross-validation (Elbow Method)
k_range = range(1, 30)
cv_scores = []
for k in k_range:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores = cross_val_score(model, X_tr, y_tr, cv=5)
    cv_scores.append(scores.mean())

best_k = k_range[np.argmax(cv_scores)]
print(f"\nBest k from CV: {best_k}, CV Score: {max(cv_scores):.4f}")

plt.plot(k_range, cv_scores, 'bo-')
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k'); plt.ylabel('CV Accuracy'); plt.title('Elbow Method for kNN')
plt.legend(); plt.show()

# Retrain with best k
best_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k))
best_knn.fit(X_tr, y_tr)
print(f"\nBest kNN test score: {best_knn.score(X_te, y_te):.4f}")
print(classification_report(y_te, best_knn.predict(X_te)))

---
<a id='12'></a>
## 12. Decision Trees

### Theory
**"A high-stakes game of 20 Questions learned from data."**

Learns a series of **if/else rules** from data. At each node it finds the best split — the one that creates the **purest** child nodes.

### Structure
- **Root node** — first split (top)
- **Internal nodes** — decision points
- **Leaf nodes** — final predictions (no further splitting)

### Impurity Measures (How to Split)
| Measure | Formula | Details |
|---------|---------|----------|
| **Gini Impurity** (default) | $G = 1 - \sum_i p_i^2$ | G=0 → pure node |
| **Entropy** | $H = -\sum_i p_i \log_2(p_i)$ | Info gain = reduction in entropy |
| **MSE** | $\frac{1}{n}\sum(y - \bar{y})^2$ | For regression trees |

Both Gini and Entropy produce similar trees in practice.

### Key Hyperparameters
| Parameter | Effect |
|-----------|--------|
| `max_depth` | Deeper = lower bias, higher variance (overfit) |
| `min_samples_leaf` | Minimum samples at a leaf — prevents tiny leaves |
| `min_samples_split` | Minimum to split an internal node |
| `criterion` | `'gini'` (default) or `'entropy'` |

### Pros & Cons
| ✅ Pros | ❌ Cons |
|---------|--------|
| Highly interpretable (white-box) | High variance — unstable |
| No feature scaling needed | Single trees overfit easily |
| Handles numeric & categorical data | Small data change → very different tree |
| Captures non-linear interactions | Rarely optimal on its own |
| Fast training | |

> Deep tree: low bias, high variance  
> Shallow tree: higher bias, lower variance

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

X_c, y_c = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Compare depths
print(f"{'Depth':<8} {'Train Score':<15} {'Val Score'}")
print("-" * 35)
for depth in [1, 2, 3, 4, 5, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_tr, y_tr)
    print(f"{str(depth):<8} {tree.score(X_tr,y_tr):<15.4f} {tree.score(X_te,y_te):.4f}")

# Best tree
best_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
best_tree.fit(X_tr, y_tr)
print("\n--- Tree Structure (depth=3) ---")
print(export_text(best_tree, feature_names=load_iris().feature_names))

print(f"Feature importances: {best_tree.feature_importances_}")
print(classification_report(y_te, best_tree.predict(X_te)))

In [ ]:
# Visualise feature importances
iris = load_iris()
feat_imp = best_tree.feature_importances_
plt.barh(iris.feature_names, feat_imp, color='steelblue')
plt.xlabel('Importance')
plt.title('Decision Tree Feature Importances')
plt.tight_layout()
plt.show()

---
<a id='13'></a>
## 13. Random Forests

### Theory
**Ensemble of decision trees — "Wisdom of the crowd."**

**How it works:**
1. Draw a **bootstrap sample** (random with replacement) from training data
2. Grow a decision tree, but at each split only consider a **random subset of features**
3. Repeat `n_estimators` times
4. **Aggregate predictions by majority vote** (classification) or average (regression)

**Why better than a single tree?**
- Each tree sees different data → different trees → **reduced variance**
- Trees that individually make bad decisions, together make good decisions
- **Out-of-bag (OOB) samples:** each tree leaves out ~1/3 of data → free internal validation estimate

### Key Hyperparameters
| Parameter | Effect |
|-----------|--------|
| `n_estimators` | Number of trees (more = better, but slower). Start 100+. |
| `max_depth` | Max depth of each tree |
| `min_samples_leaf` | Minimum samples at leaf |
| `random_state` | For reproducibility |

### Bias–Variance Summary
| Model | Bias | Variance |
|-------|------|----------|
| Deep single tree | Low | High |
| Shallow tree | Higher | Lower |
| Random forest | Low | **Low** ← best of both worlds |

### Pros & Cons
| ✅ Pros | ❌ Cons |
|---------|--------|
| Excellent generalisation | Slower than a single tree |
| Robust to overfitting | Less interpretable (black box) |
| Works great for tabular data | More memory |
| No feature scaling needed | Many hyperparameters |
| Handles missing values better | |
| Built-in feature importance | |

> **Best model for tabular data** — usually beats a single tree and often competes with neural networks.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_c, y_c = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Default RF
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
print(f"Train score: {rf.score(X_tr, y_tr):.4f}")
print(f"Test score : {rf.score(X_te, y_te):.4f}")

# Feature importances
importances = rf.feature_importances_
feat_names  = load_breast_cancer().feature_names
top10_idx   = np.argsort(importances)[-10:]
plt.figure(figsize=(8, 5))
plt.barh(feat_names[top10_idx], importances[top10_idx], color='forestgreen')
plt.xlabel('Feature Importance')
plt.title('Random Forest — Top 10 Feature Importances')
plt.tight_layout()
plt.show()

print("\n--- Classification Report ---")
print(classification_report(y_te, rf.predict(X_te), target_names=['Malignant', 'Benign']))

# GridSearch for best hyperparameters
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10]
}
gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, n_jobs=-1)
gs.fit(X_tr, y_tr)
print(f"\nBest params: {gs.best_params_}")
print(f"Best CV score: {gs.best_score_:.4f}")
print(f"Test score (best model): {gs.best_estimator_.score(X_te, y_te):.4f}")

---
<a id='14'></a>
## 14. Support Vector Machines (SVM)

### Theory
**"Find the widest possible gap (margin) between classes."**

A **hyperplane** separates classes: $w^Tx + b = 0$
- 1D → threshold
- 2D → line
- 3D → plane
- nD → hyperplane

Sign of $w^Tx + b$ determines which side of the boundary a point is on.

### Support Vectors
The **closest data points** to the decision boundary. Only these matter — moving any other point doesn't change the boundary.

### Margin
$$\text{margin} = \frac{2}{\|w\|}$$

We want to **maximise the margin** (maximise $\frac{2}{\|w\|}$, i.e. minimise $\|w\|^2$).

### Hard vs Soft Margin
| Type | Description | When to use |
|------|-------------|-------------|
| **Hard Margin** | Zero misclassifications allowed | Perfectly separable, noise-free data |
| **Soft Margin** | Allows some violations (slack variables $\xi_i$) | Real-world noisy data |

**Soft margin objective:**
$$\min_{w,b} \|w\|^2 + C\sum_i \xi_i \quad \text{s.t.} \quad y^{(i)}(w^Tx^{(i)} + b) \geq 1 - \xi_i$$

### C Parameter (Regularisation)
| C value | Margin | Violations | Risk |
|---------|--------|-----------|------|
| Small C | Large | Many allowed | Underfitting |
| Large C | Narrow | Few allowed | Overfitting |
| C → ∞ | Hard margin | None | Overfit to noise |

### Pros & Cons
| ✅ Pros | ❌ Cons |
|---------|--------|
| Works well in high dimensions | Slow on large datasets |
| Effective with clear margin | Must tune C |
| Memory efficient (only support vectors) | Feature scaling required |
| Strong baseline for high-dim data | Not directly probabilistic |
| | Hard to interpret |

In [ ]:
from sklearn.svm import SVC

X_c, y_c = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# --- Linear SVM (good baseline for high-dim data) ---
svm_small_c = make_pipeline(StandardScaler(), SVC(kernel='linear', C=0.01))
svm_large_c = make_pipeline(StandardScaler(), SVC(kernel='linear', C=10))
svm_small_c.fit(X_tr, y_tr)
svm_large_c.fit(X_tr, y_tr)
print(f"SVM C=0.01 : Train={svm_small_c.score(X_tr,y_tr):.4f}, Test={svm_small_c.score(X_te,y_te):.4f}")
print(f"SVM C=10   : Train={svm_large_c.score(X_tr,y_tr):.4f}, Test={svm_large_c.score(X_te,y_te):.4f}")

# --- Compare C values ---
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
train_scores = []
test_scores  = []
for c in C_values:
    m = make_pipeline(StandardScaler(), SVC(kernel='linear', C=c))
    m.fit(X_tr, y_tr)
    train_scores.append(m.score(X_tr, y_tr))
    test_scores.append(m.score(X_te, y_te))

plt.semilogx(C_values, train_scores, 'bo-', label='Train')
plt.semilogx(C_values, test_scores, 'rs-', label='Test')
plt.xlabel('C value'); plt.ylabel('Accuracy')
plt.title('SVM: Effect of C on Train vs Test Accuracy')
plt.legend(); plt.show()

# --- GridSearch for best C ---
gs = GridSearchCV(
    make_pipeline(StandardScaler(), SVC(kernel='linear')),
    param_grid={'svc__C': [0.001, 0.01, 0.1, 1, 2, 5, 10]},
    cv=5
)
gs.fit(X_tr, y_tr)
print(f"\nBest C: {gs.best_params_}")
print(f"Best CV: {gs.best_score_:.4f}")
print(f"Test score: {gs.score(X_te, y_te):.4f}")

---
<a id='15'></a>
## 15. Multi-class Strategies

Some models (kNN, Random Forest, Naive Bayes) **natively** support multi-class.  
Others (SVM, Logistic Regression) are **inherently binary** and need a strategy.

### One-vs-Rest (OvR) / One-vs-All
- Train **k classifiers**, one per class (class k vs all others)
- Predict: run all k classifiers, pick the class with **highest score**
- **k classifiers** total
- Default for modern SVM
- Can suffer from class imbalance

### One-vs-One (OvO)
- Train a classifier for **every pair** of classes
- **k(k-1)/2 classifiers** total
- Predict by **majority vote** across all pair classifiers
- Historical default for libSVM
- More classifiers, but each trained on fewer data

### Softmax (Multinomial Logistic Regression)
- **Single model** that directly models all k classes
- Output = probability distribution over all classes via softmax:
$$P(y=k|x) = \frac{e^{w_k^T x}}{\sum_j e^{w_j^T x}}$$
- Used in: modern `LogisticRegression`, last layer of Neural Networks

| Strategy | Classifiers | Notes |
|----------|------------|-------|
| OvR | k | Simple, widely used, default for SVM |
| OvO | k(k-1)/2 | More classifiers, each binary |
| Softmax | 1 | Single model, modern LR default |

In [ ]:
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier

X_mc, y_mc = load_iris(return_X_y=True)  # 3 classes
X_tr, X_te, y_tr, y_te = train_test_split(X_mc, y_mc, test_size=0.2, random_state=42)

# OvR
ovr = make_pipeline(StandardScaler(),
                    OneVsRestClassifier(SVC(kernel='linear')))
ovr.fit(X_tr, y_tr)
print(f"OvR SVM accuracy: {ovr.score(X_te, y_te):.4f}")

# OvO
ovo = make_pipeline(StandardScaler(),
                    OneVsOneClassifier(SVC(kernel='linear')))
ovo.fit(X_tr, y_tr)
print(f"OvO SVM accuracy: {ovo.score(X_te, y_te):.4f}")

# Softmax (modern LogisticRegression handles multi-class natively)
lr_mc = make_pipeline(StandardScaler(),
                      LogisticRegression(multi_class='multinomial', max_iter=1000))
lr_mc.fit(X_tr, y_tr)
print(f"Softmax LR accuracy: {lr_mc.score(X_te, y_te):.4f}")

# sklearn SVC with kernel linear handles OvR automatically
auto_svc = make_pipeline(StandardScaler(), SVC(kernel='linear'))
auto_svc.fit(X_tr, y_tr)
print(f"Auto SVC (OvR default): {auto_svc.score(X_te, y_te):.4f}")

---
<a id='16'></a>
## 16. GridSearchCV — Hyperparameter Tuning

**Systematically tries every combination of hyperparameters and uses k-fold CV to find the best.**

```python
GridSearchCV(estimator, param_grid, cv=5, scoring='accuracy')
```

- `estimator` — the model (or pipeline)
- `param_grid` — dict of hyperparameters to try
- `cv` — number of folds
- `n_jobs=-1` — use all CPU cores
- `.best_params_` — best hyperparameters found
- `.best_score_` — best CV score
- `.best_estimator_` — the best model (already trained)

**For pipelines:** prefix parameter names with `stepname__` (double underscore).

> Remember: GridSearch uses cross-validation internally, so it's fine to use `X_train` only.

In [ ]:
from sklearn.model_selection import GridSearchCV

X_c, y_c = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# --- GridSearch for kNN ---
knn_pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
param_grid_knn = {'kneighborsclassifier__n_neighbors': list(range(1, 21, 2))}
gs_knn = GridSearchCV(knn_pipe, param_grid_knn, cv=5, n_jobs=-1)
gs_knn.fit(X_tr, y_tr)
print("=== kNN GridSearch ===")
print(f"Best params: {gs_knn.best_params_}")
print(f"Best CV score: {gs_knn.best_score_:.4f}")
print(f"Test score: {gs_knn.score(X_te, y_te):.4f}")

# --- GridSearch for SVM ---
svm_pipe = make_pipeline(StandardScaler(), SVC(kernel='linear'))
param_grid_svm = {'svc__C': [0.001, 0.01, 0.1, 1, 10, 100]}
gs_svm = GridSearchCV(svm_pipe, param_grid_svm, cv=5, n_jobs=-1)
gs_svm.fit(X_tr, y_tr)
print("\n=== SVM GridSearch ===")
print(f"Best params: {gs_svm.best_params_}")
print(f"Best CV score: {gs_svm.best_score_:.4f}")
print(f"Test score: {gs_svm.score(X_te, y_te):.4f}")

# --- GridSearch for Random Forest ---
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [1, 5]
}
gs_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, n_jobs=-1)
gs_rf.fit(X_tr, y_tr)
print("\n=== Random Forest GridSearch ===")
print(f"Best params: {gs_rf.best_params_}")
print(f"Best CV score: {gs_rf.best_score_:.4f}")
print(f"Test score: {gs_rf.score(X_te, y_te):.4f}")

---
<a id='17'></a>
## 17. K-Means Clustering (Unsupervised)

### Theory
**Unsupervised** — no labels. Finds `k` natural groups in data.

**Algorithm:**
1. Randomly place `k` centroid points
2. Assign each data point to its nearest centroid
3. Move each centroid to the mean of its assigned points
4. Repeat until centroids stop moving

**Distance:** Euclidean (so scaling is essential)

### Choosing k
- **Elbow Method:** plot inertia (within-cluster sum of squares) vs k — look for the "elbow"

### Pros & Cons
| ✅ Pros | ❌ Cons |
|---------|--------|
| Simple, fast | Must choose k in advance |
| Scales to large data | Assumes spherical clusters |
| No labels needed | Sensitive to scale and outliers |
| | Different runs can give different results |

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Create synthetic data
from sklearn.datasets import make_blobs
X_km, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Scale first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_km)

# Fit K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X_scaled)
labels = kmeans.labels_
centroids = kmeans.cluster_centers_

# Plot clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='viridis', alpha=0.7)
axes[0].scatter(centroids[:, 0], centroids[:, 1], s=300, c='red', marker='X', label='Centroids')
axes[0].set_title('K-Means (k=4)')
axes[0].legend()

# Elbow Method
inertias = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

axes[1].plot(k_range, inertias, 'bo-')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Inertia (WCSS)')
axes[1].set_title('Elbow Method — Choose k at the "elbow"')

plt.tight_layout()
plt.show()

print(f"Inertia (k=4): {kmeans.inertia_:.2f}")
print(f"Labels: {np.unique(labels)}")

---
<a id='18'></a>
## 18. Complete Workflow Template

This is the full pipeline from the Digits Lab (Week 6) generalised for any classification problem.

In [ ]:
# ============================================================
# COMPLETE ML CLASSIFICATION WORKFLOW TEMPLATE
# (Generalised from Digits Lab Week 6)
# ============================================================
from sklearn.datasets import load_digits  # Using sklearn's built-in digits
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# --- STEP 1: Load & Prepare Data ---
digits = load_digits()
X = digits.data / 16.0   # Scale pixel values to [0, 1]
y = digits.target
print(f"X shape: {X.shape}  y shape: {y.shape}")
print(f"Classes: {np.unique(y)}")

# --- STEP 2: Train / Validation / Test Split (60/20/20) ---
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# --- STEP 3: Train Baseline Models ---
results = {}

# Linear SVM
svm = make_pipeline(StandardScaler(), SVC(kernel='linear', C=1))
svm.fit(X_train, y_train)
results['SVM (linear, C=1)'] = {
    'train': svm.score(X_train, y_train),
    'val': svm.score(X_val, y_val)
}

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
results['Random Forest (default)'] = {
    'train': rf.score(X_train, y_train),
    'val': rf.score(X_val, y_val)
}

# kNN
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_train)
results['kNN (k=5)'] = {
    'train': knn.score(X_train, y_train),
    'val': knn.score(X_val, y_val)
}

# Logistic Regression
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
lr.fit(X_train, y_train)
results['Logistic Regression'] = {
    'train': lr.score(X_train, y_train),
    'val': lr.score(X_val, y_val)
}

print(f"{'Model':<30} {'Train Score':<15} {'Val Score':<15} {'Overfit?'}")
print("-" * 65)
for name, scores in results.items():
    gap     = scores['train'] - scores['val']
    overfit = 'YES' if gap > 0.05 else 'No'
    print(f"{name:<30} {scores['train']:<15.4f} {scores['val']:<15.4f} {overfit}")

In [ ]:
# --- STEP 4: Hyperparameter Tuning with GridSearchCV ---
# Tune SVM C parameter using 5-fold CV on training set
gs_svm = GridSearchCV(
    make_pipeline(StandardScaler(), SVC(kernel='linear')),
    param_grid={'svc__C': [0.001, 0.01, 0.1, 1, 10]},
    cv=5, n_jobs=-1
)
gs_svm.fit(X_train, y_train)
print(f"SVM Best C: {gs_svm.best_params_}")
print(f"SVM Best CV: {gs_svm.best_score_:.4f}")
print(f"SVM Val score (best): {gs_svm.score(X_val, y_val):.4f}")

# Tune Random Forest
gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20]
    },
    cv=5, n_jobs=-1
)
gs_rf.fit(X_train, y_train)
print(f"\nRF Best params: {gs_rf.best_params_}")
print(f"RF Best CV: {gs_rf.best_score_:.4f}")
print(f"RF Val score (best): {gs_rf.score(X_val, y_val):.4f}")

In [ ]:
# --- STEP 5: Model Selection — Pick Best Validation Score ---
# Suppose SVM wins. Combine train+val, retrain, evaluate ONCE on test.

best_C = gs_svm.best_params_['svc__C']
final_model = make_pipeline(StandardScaler(), SVC(kernel='linear', C=best_C))

# Combine train + val
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.concatenate([y_train, y_val])

final_model.fit(X_train_val, y_train_val)

# --- STEP 6: Final Evaluation on Test Set (ONE TIME ONLY) ---
y_pred_final = final_model.predict(X_test)
print(f"FINAL TEST ACCURACY: {accuracy_score(y_test, y_pred_final):.4f}")
print("\n--- Final Classification Report ---")
print(classification_report(y_test, y_pred_final))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_final)
).plot(ax=ax, colorbar=True)
ax.set_title(f'Final Confusion Matrix — SVM Linear C={best_C}')
plt.tight_layout()
plt.show()

---
<a id='19'></a>
## 19. Quick Reference — Pros & Cons Summary Table

| Model | Type | Scaling Needed? | Interpretable? | Best For | Avoid When |
|-------|------|----------------|---------------|----------|------------|
| **Linear Regression** | Regression | No* | ✅ Yes | Continuous output, linear relationships | Non-linear data |
| **Ridge/Lasso** | Regression | Yes | ✅ Yes | Overfitting control, feature selection (Lasso) | — |
| **Logistic Regression** | Classification | Yes | ✅ Yes | Binary/multi-class, simple baseline | Non-linear boundaries |
| **kNN** | Both | ✅ Required | ⚠️ Medium | Simple patterns, small datasets | High dim, large datasets |
| **Decision Tree** | Both | No | ✅ Yes | Interpretability, tree rules visible | Alone — use RF instead |
| **Random Forest** | Both | No | ⚠️ Medium | Tabular data, most general | Image/text/audio data |
| **SVM (Linear)** | Classification | ✅ Required | ❌ No | High-dim text/image data | Very large datasets |
| **Naive Bayes** | Classification | No | ✅ Yes | Text classification, fast baseline | Correlated features |
| **K-Means** | Clustering | ✅ Required | ✅ Yes | Finding natural groups (unsupervised) | Need exact k, non-spherical clusters |

## Key Formulas Cheat Sheet

| Concept | Formula |
|---------|--------|
| Linear model | $\hat{y} = w_0 + w_1x_1 + \ldots + w_nx_n$ |
| Sigmoid | $\sigma(x) = \frac{1}{1+e^{-x}}$ |
| MSE | $\frac{1}{m}\sum(\hat{y}_i - y_i)^2$ |
| Gini | $G = 1 - \sum_i p_i^2$ |
| SVM Margin | $M = \frac{2}{\|w\|}$ |
| L2 Reg | $L(w) = \text{MSE}(w) + \lambda\sum w_j^2$ |
| Precision | $\frac{TP}{TP+FP}$ |
| Recall | $\frac{TP}{TP+FN}$ |
| F1 | $2 \cdot \frac{P \cdot R}{P+R}$ |
| Accuracy | $\frac{TP+TN}{TP+FP+TN+FN}$ |
| Euclidean dist | $d = \sqrt{\sum_i (t_i - s_i)^2}$ |

## Decision Guide: Which Model to Use?

```
START
 ├─ Do you have labels (y)? 
 │   ├─ NO  → K-Means (unsupervised clustering)
 │   └─ YES (supervised)
 │       ├─ Is y continuous? → Linear Regression / Ridge / Lasso
 │       └─ Is y categorical? (classification)
 │           ├─ Need interpretability?     → Decision Tree / Logistic Regression
 │           ├─ Tabular, general purpose?  → Random Forest ← best default
 │           ├─ High-dimensional?          → SVM (linear kernel)
 │           ├─ Small dataset, simple?     → kNN (with scaling!)
 │           └─ Fast baseline?             → Naive Bayes / Logistic Regression
```

## Common Mistakes to Avoid

1. ❌ **Using test set during model selection** — contaminates it, gives false confidence
2. ❌ **Fitting scaler on test data** — use pipelines, fit only on train
3. ❌ **Forgetting to scale for kNN/SVM/k-Means** — distance metrics are scale-sensitive
4. ❌ **Using MSE loss for Logistic Regression** — use log-loss instead
5. ❌ **Single train/test split for hyperparameter selection** — use cross-validation
6. ❌ **High training accuracy = good model** — check generalisation on validation/test set
7. ❌ **Extrapolating beyond training data range** — risky and unreliable

In [ ]:
# =============================================
# QUICK REFERENCE: All Models in One Cell
# =============================================
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Logistic Regression' : make_pipeline(StandardScaler(), LogisticRegression(max_iter=200)),
    'kNN (k=5)'           : make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Decision Tree (d=3)' : DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (linear)'        : make_pipeline(StandardScaler(), SVC(kernel='linear', C=1)),
    'Naive Bayes'         : GaussianNB(),
}

print(f"{'Model':<25} {'Train':>8} {'Test':>8}")
print("-" * 44)
for name, m in models.items():
    m.fit(X_train, y_train)
    tr = m.score(X_train, y_train)
    te = m.score(X_test, y_test)
    print(f"{name:<25} {tr:>8.4f} {te:>8.4f}")